# 1.7 Profil ve Zamanlama

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/01-ipython/07-timing-and-profiling.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Profiling and Timing Code

Kod geliştirirken ve veri işleme hatları kurarken çeşitli uygulamalar arasında ödünleşimler yapmanız gerekir. Algoritmanızı geliştirirken erken aşamada bunları düşünmek genelde verimsizdir. Donald Knuth'un ünlü sözüyle: “Küçük verimlilikleri, zamanın yaklaşık %97'sinde unutmalıyız: erken optimizasyon tüm kötülüklerin köküdür.”

Ancak kodunuz çalıştığında verimliliğine biraz bakmak faydalı olabilir. Bazen tek bir komutun veya komut kümesinin yürütme süresini kontrol etmek; bazen de çok adımlı bir süreçte darboğazın nerede olduğunu görmek gerekir. IPython bu tür zamanlama ve profilleme için geniş bir işlevsellik sunar. Burada şu IPython magic komutlarını ele alacağız:

Son dört komut IPython ile birlikte gelmez; kullanmak için line_profiler ve memory_profiler eklentilerini kurmanız gerekir — aşağıdaki bölümlerde bunları ele alacağız.

## Kod parçalarını zamanlama: %timeit ve %time

1.3 Magic Komutlar bölümünde %timeit satır magic'i ve %%timeit hücre magic'ini görmüştük; bunlar kod parçalarının tekrarlı yürütmesini zamanlamak için kullanılır:


```python
# timeit_sum.ipynb
%timeit sum(range(100))
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


İşlem çok hızlı olduğu için %timeit otomatik olarak çok sayıda tekrar yapar. Daha yavaş komutlarda %timeit otomatik ayarlanır ve daha az tekrar yapar:


```python
# timeit_cell.ipynb
%%timeit
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Bazen bir işlemi tekrarlamak en iyi seçenek değildir. Örneğin sıralamak istediğimiz bir listede yanıltılabiliriz; önceden sıralanmış bir listeyi sıralamak sıralanmamış listeden çok daha hızlıdır, bu yüzden tekrar sonucu çarpıtır:


```python
# timeit_sort.ipynb
import random
L = [random.random() for i in range(100000)]
%timeit L.sort()
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Bunun için %time magic fonksiyonu daha iyi bir seçenek olabilir. Kısa sistem gecikmelerinin sonucu etkilemeyeceği uzun süren komutlar için de uygundur. Sıralanmamış ve önceden sıralanmış bir listeyi zamanlayalım:


```python
# time_sort1.ipynb
import random
L = [random.random() for i in range(100000)]
print("sorting an unsorted list:")
%time L.sort()
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


```python
# time_sort2.ipynb
print("sorting an already sorted list:")
%time L.sort()
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Önceden sıralanmış listenin ne kadar daha hızlı sıralandığına dikkat edin; aynı zamanda önceden sıralanmış liste için bile %time'ın %timeit'e kıyasla zamanlamanın ne kadar daha uzun sürdüğüne bakın! Bunun nedeni %timeit'in zamanlamayı bozabilecek sistem çağrılarını engellemek için arka planda akıllı işler yapmasıdır. Örneğin kullanılmayan Python nesnelerinin temizlenmesini (garbage collection) engelleyebilir. Bu yüzden %timeit sonuçları genelde %time sonuçlarından belirgin şekilde daha hızlıdır.

%time için de %timeit gibi %% hücre magic sözdizimi çok satırlı betikleri zamanlamaya izin verir:


```python
# time_cell.ipynb
%%time
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


%time ve %timeit hakkında daha fazla bilgi ve seçenekler için IPython yardımını kullanın (örneğin IPython isteminde %time? yazın).

> **Not**
>

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin — perf_counter
      Python sum ile NumPy np.sum süresini karşılaştırın:
      
        import time
import numpy as np

big = np.random.rand(1_000_000)

t0 = time.perf_counter()
sum(big)
t1 = time.perf_counter()
np.sum(big)
t2 = time.perf_counter()

print(f"Python sum: {(t1-t0)*1000:.1f} ms")
print(f"NumPy sum:  {(t2-t1)*1000:.2f} ms")

## Tam betik profilleme: %prun

Bir program birçok tek ifadeden oluşur; bazen bu ifadeleri bağlam içinde zamanlamak, tek başına zamanlamaktan daha önemlidir. Python yerleşik bir kod profiler'ı içerir (Python dokümantasyonunda okuyabilirsiniz); IPython bunu %prun magic fonksiyonu biçiminde çok daha kullanışlı sunar.

Örnek olarak bazı hesaplamalar yapan basit bir fonksiyon tanımlayalım:


In [ ]:
# sum_of_lists.py
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total



Profil sonuçlarını görmek için %prun'u bir fonksiyon çağrısıyla çağırabiliriz:


```python
# prun.ipynb
%prun sum_of_lists(1000000)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Sonuç, her fonksiyon çağrısında toplam süreye göre sıralanmış bir tablodur; yürütmenin en çok zamanı nerede harcadığını gösterir. Bu örnekte yürütme süresinin büyük kısmı sum_of_lists içindeki liste üretecindedir. Buradan algoritmanın performansını iyileştirmek için hangi değişiklikleri düşünebileceğimize başlayabiliriz.

%prun ve seçenekleri hakkında daha fazla bilgi için IPython yardımını kullanın (IPython isteminde %prun?).

## %lprun ile satır satır profilleme

%prun'un fonksiyon bazlı profillemesi yararlıdır; ancak bazen satır satır profil raporu daha uygundur. Bu Python veya IPython'a gömülü değildir; kurulabilir line_profiler paketi vardır. Python paketleme aracı pip ile kurun:


```
$ pip install line_profiler
```


Ardından IPython ile bu paketin parçası olan line_profiler uzantısını yükleyin:


```python
# load_line_profiler.ipynb
%load_ext line_profiler
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Artık %lprun herhangi bir fonksiyonu satır satır profiller. Bu örnekte hangi fonksiyonları profillemek istediğimizi açıkça belirtmemiz gerekir:


```python
# lprun.ipynb
%lprun -f sum_of_lists sum_of_lists(5000)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Üstteki bilgi sonuçları okumanın anahtarıdır: süre mikrosaniye cinsinden raporlanır; programın en çok zamanı nerede harcadığını görebiliriz. Bu noktada betiği istediğimiz kullanım için daha iyi performans gösterecek şekilde değiştirebiliriz.

%lprun ve seçenekleri için IPython yardımını kullanın (%lprun?).

## Bellek kullanımını profilleme: %memit ve %mprun

Profillemenin bir başka yönü bir işlemin ne kadar bellek kullandığıdır. Bunu memory_profiler adlı başka bir IPython uzantısıyla değerlendirebilirsiniz. line_profiler gibi önce pip ile kurun:


```
$ pip install memory_profiler
```


Ardından IPython ile yükleyin:


```python
# load_memory_profiler.ipynb
%load_ext memory_profiler
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Bellek profiler uzantısı iki yararlı magic içerir: %memit (%timeit'in bellek ölçen karşılığı) ve %mprun (%lprun'un bellek ölçen karşılığı). %memit oldukça basit kullanılır:


```python
# memit.ipynb
%memit sum_of_lists(1000000)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Bu fonksiyonun yaklaşık 140 MB bellek kullandığını görürüz.

Bellek kullanımının satır satır açıklaması için %mprun kullanılabilir. Ne yazık ki bu yalnızca notebook'un kendisinde değil, ayrı modüllerde tanımlı fonksiyonlar için çalışır; bu yüzden %%file hücre magic'i ile mprun_demo.py adlı basit bir modül oluşturarak sum_of_lists fonksiyonumuzu (bellek profili sonuçlarını netleştirmek için del L eklenmiş haliyle) yazacağız:


```python
# mprun_demo.py
%%file mprun_demo.py
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
        del L # remove reference to L
    return total
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Şimdi bu fonksiyonun yeni sürümünü içe aktarıp bellek satır profiler'ını çalıştırabiliriz:


```python
# mprun.ipynb
from mprun_demo import sum_of_lists
%mprun -f sum_of_lists sum_of_lists(1000000)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Burada Increment sütunu her satırın toplam bellek bütçesini ne kadar etkilediğini söyler: L listesini oluşturup sildiğimizde yaklaşık 30 MB bellek kullanımı eklenir. Bu, Python yorumlayıcısının kendi arka plan bellek kullanımının üzerinedir.

%memit ve %mprun hakkında daha fazla bilgi için IPython yardımını kullanın (örneğin %memit?).

> **Not**
>
